# Лабораторная работа 4. Часть 2 — Т-тесты

**ФИО:** Иноземцев Сергей Викторович  
**Группа:** ЕТ-216

Цель: проверить различия средних психологических показателей между юношами и девушками. Исходный файл называется `Психологи.xlsx` (в тексте задания — «Психология.xlsx»).

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display, Markdown

pd.set_option('display.max_columns', 25)
pd.set_option('display.max_rows', 25)
sns.set_theme(style='whitegrid', font='DejaVu Sans')
plt.rcParams.update({'figure.figsize': (9, 5), 'figure.dpi': 110})
DATA = Path('.')
FIG = Path('figures')
FIG.mkdir(exist_ok=True)

def show(name):
    plt.tight_layout()
    plt.savefig(FIG / (name + '.png'), bbox_inches='tight')
    plt.show()
    plt.close('all')


## 2. Гипотезы и выбор критерия

H₀: средние значения показателя у юношей и девушек одинаковы. H₁: средние различаются. Уровень значимости α=0,05. Группы независимы; используем двусторонний t-тест Уэлча (`equal_var=False`). Парный тест не подходит, поскольку строки не являются повторными измерениями одних людей.

Номер учащегося и код профиля исключены из психологических шкал. Для каждого из 15 показателей приведены t, исходное p-value и вывод при α=0,05; отдельно добавлена поправка Holm на множественные проверки, 95% ДИ разности и Cohen d. Неотклонение H₀ не доказывает равенство групп.

In [2]:
def holm(pvalues):
    p=np.asarray(pvalues);order=np.argsort(p);adjusted=np.empty(len(p))
    adjusted[order]=np.minimum(1,np.maximum.accumulate((len(p)-np.arange(len(p)))*p[order]));return adjusted
def compare_groups(frame,group,cols,a,b):
    rows=[]
    for col in cols:
        x=frame.loc[frame[group].eq(a),col].dropna().to_numpy();y=frame.loc[frame[group].eq(b),col].dropna().to_numpy()
        t,p=stats.ttest_ind(x,y,equal_var=False)
        diff=x.mean()-y.mean();vx=x.var(ddof=1)/len(x);vy=y.var(ddof=1)/len(y);se=np.sqrt(vx+vy)
        dof=(vx+vy)**2/(vx*vx/(len(x)-1)+vy*vy/(len(y)-1));margin=stats.t.ppf(.975,dof)*se
        pooled=np.sqrt(((len(x)-1)*x.var(ddof=1)+(len(y)-1)*y.var(ddof=1))/(len(x)+len(y)-2))
        rows.append([col,len(x),len(y),x.mean(),y.mean(),diff,t,p,diff-margin,diff+margin,diff/pooled])
    r=pd.DataFrame(rows,columns=['показатель','n_A','n_B','mean_A','mean_B','A-B','t','p','CI_low','CI_high','Cohen_d'])
    r['значимо без поправки']=r.p<.05;r['p_Holm']=holm(r.p);r['вывод_0.05']=np.where(r.p_Holm<.05,'H0 отклонена','H0 не отклонена');return r


## Психологические показатели

In [3]:
psy=pd.read_excel(DATA/'Психологи.xlsx');psy.columns=psy.columns.str.strip();psy['пол']=psy['пол'].str.strip().str.lower()
display(psy.head());display(psy['пол'].value_counts());display(psy.isna().sum())
psy_cols=[c for c in psy.select_dtypes('number').columns if c not in ['учащиеся','профиль ВУЗа']]
psy_tests=compare_groups(psy,'пол',psy_cols,'м','ж');display(psy_tests.sort_values('p'))
normal_psy=[]
for c in psy_cols:
    for g in ['м','ж']:
        x=psy.loc[psy['пол'].eq(g),c].dropna();normal_psy.append([c,g,len(x),stats.shapiro(x).pvalue])
display(pd.DataFrame(normal_psy,columns=['шкала','пол','n','p_Shapiro']))
psy_tests.to_csv('psychology_welch.csv',index=False)

,учащиеся,пол,класс,профиль ВУЗа,осведомленность,скрытые фигуры,пропущенные слова,арифметика,понятливость,исключение изображений,аналогии,числовые ряды,умозаключения,геометрическое сложение,заучивание слов,средний IQ,экстраверсия- интроверсия,нейротизм,средняя отметка
0,1,ж,гум,0,12,9,11,8,8,11,13,8,12,10,11,103,15,7,3.93
1,2,ж,гум,1,10,12,12,11,10,12,9,12,8,11,11,107,13,17,4.27
2,3,м,мат,1,11,8,9,11,11,12,9,11,9,11,8,100,10,19,3.87
3,4,ж,гум,0,14,12,14,13,13,9,9,9,12,11,12,116,13,11,4.57
4,5,ж,гум,0,12,12,9,10,9,12,5,10,3,11,9,93,16,8,4.14


пол
ж    23
м    23
Name: count, dtype: int64

учащиеся                     0
пол                          0
класс                        0
профиль ВУЗа                 0
осведомленность              0
скрытые фигуры               0
пропущенные слова            0
арифметика                   0
понятливость                 0
исключение  изображений      0
аналогии                     0
числовые ряды                0
умозаключения                0
геометрическое  сложение     0
заучивание слов              0
средний IQ                   0
экстраверсия- интроверсия    0
нейротизм                    0
средняя отметка              0
dtype: int64

,показатель,n_A,n_B,mean_A,mean_B,A-B,t,p,CI_low,CI_high,Cohen_d,значимо без поправки,p_Holm,вывод_0.05
12,экстраверсия- интроверсия,23,23,11.565217,14.608696,-3.043478,-3.444096,0.001313,-4.826926,-1.260030,-1.015608,True,0.019699,H0 отклонена
1,скрытые фигуры,23,23,9.043478,10.608696,-1.565217,-2.011319,0.050646,-3.135039,0.004604,-0.593106,False,0.709042,H0 не отклонена
3,арифметика,23,23,11.695652,10.695652,1.000000,1.668595,0.102715,-0.209826,2.209826,0.492042,False,1.000000,H0 не отклонена
13,нейротизм,23,23,12.826087,11.000000,1.826087,1.614516,0.113599,-0.453687,4.105861,0.476095,False,1.000000,H0 не отклонена
7,числовые ряды,23,23,10.434783,9.347826,1.086957,1.602253,0.116539,-0.281791,2.455704,0.472479,False,1.000000,H0 не отклонена
10,заучивание слов,23,23,9.173913,10.173913,-1.000000,-1.535506,0.132067,-2.313799,0.313799,-0.452796,False,1.000000,H0 не отклонена
14,средняя отметка,23,23,3.976957,4.151304,-0.174348,-1.489710,0.143474,-0.410253,0.061558,-0.439292,False,1.000000,H0 не отклонена
0,осведомленность,23,23,10.304348,11.260870,-0.956522,-1.473229,0.148103,-2.266581,0.353537,-0.434432,False,1.000000,H0 не отклонена
5,исключение изображений,23,23,9.304348,10.260870,-0.956522,-1.357282,0.182584,-2.382505,0.469461,-0.400241,False,1.000000,H0 не отклонена
9,геометрическое сложение,23,23,11.608696,10.608696,1.000000,1.337259,0.188207,-0.508315,2.508315,0.394336,False,1.000000,H0 не отклонена


,шкала,пол,n,p_Shapiro
0,осведомленность,м,23,0.154013
1,осведомленность,ж,23,0.454872
2,скрытые фигуры,м,23,0.198462
3,скрытые фигуры,ж,23,0.003562
4,пропущенные слова,м,23,0.557954
...,...,...,...,...
25,экстраверсия- интроверсия,ж,23,0.053863
26,нейротизм,м,23,0.187958
27,нейротизм,ж,23,0.877465
28,средняя отметка,м,23,0.323701


Минимальное p у шкалы «экстраверсия- интроверсия», максимальное — «средний IQ». Это сила статистического свидетельства, не размер различия: величина эффекта отдельно дана в Cohen d. После Holm значимых шкал 1 из 15. При малой выборке, ненормальности или порядковых шкалах выводы t-тестов требуют осторожности; группы могут различаться также по классу и профилю, поэтому причинный эффект пола не установлен.

## Итоговый вывод
Статистику и p-value показываем для каждой шкалы. Наиболее и наименее выраженные различия сравниваем по абсолютному стандартизованному эффекту Cohen d, чтобы шкалы с разными единицами были сопоставимы.

In [4]:
for label, idx in [('Сильнее всего', psy_tests.Cohen_d.abs().idxmax()),
                   ('Слабее всего', psy_tests.Cohen_d.abs().idxmin())]:
    row = psy_tests.loc[idx]
    print(f"{label}: {row['показатель']}; средние юношей/девушек "
          f"{row.mean_A:.3f}/{row.mean_B:.3f}; d={row.Cohen_d:.3f}; "
          f"t={row.t:.4f}; p={row.p:.6g}.")
print('Значимые шкалы до поправки:', ', '.join(psy_tests.loc[psy_tests.p < .05, 'показатель']))
print('Значимые шкалы после Holm:', ', '.join(psy_tests.loc[psy_tests.p_Holm < .05, 'показатель']))


Сильнее всего: экстраверсия- интроверсия; средние юношей/девушек 11.565/14.609; d=-1.016; t=-3.4441; p=0.00131324.
Слабее всего: средний IQ; средние юношей/девушек 104.522/105.217; d=-0.049; t=-0.1657; p=0.86916.
Значимые шкалы до поправки: экстраверсия- интроверсия
Значимые шкалы после Holm: экстраверсия- интроверсия
